In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 连续角点：固定八条补充

本包需经控制会话批准后手动运行。固定两图×±15°×CG/G，黑色双线性，只取消256网格角点round。
保留128尺度/偏置、255归一、原H求解与PIL恢复，测当前完整17W/m。不是搜索，不按truth选择H或调整tau。

CG raw复用已审64测量中随代码携带的记录；G raw读取上一轮52结果。**不再检测几何，不重跑原52参照，不生成或嵌入。**
主tau=1.2657276026437319仅描述。G含同步、不含内容，不能报告正式FPR。旧失败与所有八条结果保留。

## 1. 固定代码与只读输入
复制原R0四图/result到临时cache；读取上轮content-renderer-v1的原对象，不改写它。

In [ ]:
from pathlib import Path
import json, sys, subprocess, shutil, time
EXACT = '74470afc23b858655d516268760cc430c1905d27'
REPO = Path('/content/ceg-wm-renderer-subpixel-74470af')
R0 = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7/4f0bf1560805672f786dc86dd50d793aec18aae7/r0-f1')
CACHE = Path('/content/renderer-subpixel-inputs-74470af')
OUTPUT = Path('/content/drive/MyDrive/CEG-WM/RotationRenderer-Diagnostic-V1/content-subpixel-v1')
PREVIOUS = Path('/content/drive/MyDrive/CEG-WM/RotationRenderer-Diagnostic-V1/content-renderer-v1')
for name in ('plan.json','pilot.jsonl','remaining.jsonl'):
    if not (PREVIOUS/name).is_file(): raise FileNotFoundError('缺少上轮52路线原始结果: '+name)
if OUTPUT.exists():
    raise FileExistsError('已有内容输出，保留原结果，不自动覆盖或重跑。')
if not REPO.exists():
    subprocess.run(['git','clone','--branch','RotationRenderer-Diagnostic-V1','--single-branch','https://github.com/RICHAAARC/CEG-WM.git',str(REPO)],check=True)
elif subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
    raise RuntimeError('保留已有代码改动。')
subprocess.run(['git','-C',str(REPO),'checkout','--detach',EXACT],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',str(REPO)],check=True)
sys.path.insert(0,str(REPO))
sys.path.insert(0,str(REPO/'src'))
from diagnostics.rotation_renderer.content_plan import prepare_sources, validate_scoring_assets, UNITS, ARMS, ARM_NAMES
from diagnostics.rotation_renderer.subpixel import SubpixelSession
REFERENCE = REPO/'diagnostics/rotation_renderer/cg_geometry_reference.json'
original = json.loads((R0/'result.json').read_text(encoding='utf-8-sig'))
mapping = []
for unit in UNITS:
    records = [r for r in original['raw_unit_records'] if r['stage']=='evaluation' and r['unit_id']==unit]
    if len(records)!=1: raise ValueError('原R0单元缺失或重复')
    for arm in ARMS:
        arms = [r for r in records[0]['arms'] if r['arm']==ARM_NAMES[arm]]
        if len(arms)!=1: raise ValueError('原R0 arm缺失或重复')
        relative = arms[0]['image_file']
        source = (R0/relative).resolve()
        if not source.is_relative_to(R0.resolve()) or not source.is_file():
            raise FileNotFoundError('原CG/G路径不存在或不在R0目录内')
        mapping.append(dict(unit_id=unit,arm=arm,original_path=relative,local_filename=f'{unit}__{arm}.png'))
CACHE.mkdir(parents=True,exist_ok=False)
shutil.copyfile(R0/'result.json',CACHE/'r0-result.json')
for item in mapping:
    shutil.copyfile(R0/item['original_path'],CACHE/item['local_filename'])
(CACHE/'content_source_mapping.json').write_text(json.dumps({'images':mapping},indent=2))
renderer_r0, renderer_sources = prepare_sources(CACHE)
print({'code':EXACT,'original_pairs':len(renderer_sources)//2,'planned_routes':8,'output':str(OUTPUT)})

## 2. 同key/同资产，优先复用

优先复用上一轮renderer_assets或renderer_session.assets，类型及计算资产匹配后直接使用。
若没有匹配的既有资产，使用既有production factory初始化并单列耗时。Secret原文不输出，无A100限定。
本单元不评分；下一单元才运行八条。冷初始化可能远慢于八条评分本身。

In [ ]:
from google.colab import userdata
import torch
try:
    renderer_key = userdata.get('CEG_WM_ROOT_KEY')
except Exception:
    raise RuntimeError('请启用Colab Secret CEG_WM_ROOT_KEY访问') from None
if not renderer_key: raise RuntimeError('CEG_WM_ROOT_KEY为空')
from cegwm.shared.keys import public_key_digest
if public_key_digest(renderer_key) != renderer_r0['public_key_digest']:
    raise ValueError('Secret与原R0嵌入密钥不一致；先修正密钥，尚未初始化模型。')
setup_started = time.perf_counter()
_previous_renderer_assets = globals().get('renderer_assets')
_previous_renderer_session = globals().get('renderer_session')
renderer_assets = None
renderer_reuse = None
candidates = [(name,globals().get(name)) for name in ('production_assets','assets','runtime_assets')]
candidates.insert(0,('renderer_assets',_previous_renderer_assets))
if _previous_renderer_session is not None:
    candidates.insert(0,('renderer_session.assets',getattr(_previous_renderer_session,'assets',None)))
previous_session = globals().get('session')
if previous_session is not None:
    candidates.append(('session.assets',getattr(previous_session,'assets',None)))
for label,candidate in candidates:
    if candidate is None: continue
    try:
        validate_scoring_assets(candidate,renderer_key,renderer_r0)
    except (TypeError,ValueError,AttributeError):
        continue
    renderer_assets,renderer_reuse = candidate,label
    break
if renderer_assets is None:
    try: renderer_token = userdata.get('HF_TOKEN')
    except Exception: raise RuntimeError('请启用Colab Secret HF_TOKEN访问') from None
    if not renderer_token: raise RuntimeError('HF_TOKEN为空')
    from experiments.run_blind_detection_v1 import build_production_runtime, load_runtime_config
    runtime_root = CACHE/'runtime'
    runtime_root.mkdir(exist_ok=False)
    renderer_pipeline,renderer_assets = build_production_runtime(REPO,load_runtime_config(REPO),hf_token=renderer_token,runtime_root=runtime_root)
    del renderer_token
    renderer_reuse = 'fresh existing production factory'
renderer_session = SubpixelSession(CACHE,REFERENCE,PREVIOUS,OUTPUT,renderer_key,renderer_assets)
setup = {'initialization_seconds':time.perf_counter()-setup_started,'asset_source':renderer_reuse,
         'torch':torch.__version__,'cuda':torch.cuda.is_available(),
         'gpu':torch.cuda.get_device_name() if torch.cuda.is_available() else None,
         'geometry_device':str(renderer_assets.geometry_backend.device),'code':EXACT}
(OUTPUT/'initialization.json').write_text(json.dumps(setup,indent=2))
print(setup)

## 3. 一次运行固定八条

八条全部保留，不因过阈值与否选择候选。缺raw/非法H按失败行记录，不回退、不自动重试。
连续转换的几何误差改善不等于内容恢复；本单元才提供对应内容证据。

In [ ]:
summary = renderer_session.run()
print(json.dumps(summary,indent=2))

## 4. 查看同图对照并回传

old_post/oracle从原52记录只读引用，未重新评分。rows.jsonl保存连续角点/H、原raw几何、全部17W/m及错误。
请回传新的content-subpixel-v1目录供审计；本两图诊断不等同独立验证或正式协议成功，V2全域搜索仍暂停。

In [ ]:
import pandas as pd
table = pd.read_csv(OUTPUT/'paired.csv')
display(table)
print({'recorded_routes':len(table),'unique_routes':len(table.drop_duplicates(['unit','arm','condition']))})